In [1]:
from manim import *

video_scene = " -v WARNING --disable_caching  falling_drop_Scene"
image_scene = f" -v WARNING --disable_caching -r {2*427},{2*240}  -s  falling_drop_Scene"

In [2]:
# colors
uni_blau_1 = ManimColor.from_hex("#153268")
uni_blau_2 = ManimColor.from_hex("#005f9b")
jura_rot = ManimColor.from_hex('#c70d2d')

In [ ]:
# position y(t) given the constants
y0 = 100					# initital drop height
v0 = 0					# initial velocity

g = -10					# acceleration
m = 2					# mass
k = 1					# friction constant
tau = m/k				# time constant


# position
def get_y(t):
    return y0 - g*tau**2 + g*tau*t - tau * (v0 - g*tau) * np.exp(-t / tau) 


# velocity
def get_v(t):
    return g*tau + (v0 - g*tau) * np.exp(-t / tau) 


# gravitational force
def get_force_gravitation(t):
    return m*g


# friction force
def get_force_friction(t):
    return -k*get_v(t)

In [35]:
class FallingDrop(Mobject):
	def __init__(self, ax_center = np.array([0, 0, 0]), y_length = 5, **kwargs):
		super().__init__(**kwargs)

		self.ax_center = ax_center

		x_range = [-2, 2, 1]
		y_range = [0, int(y0), int(y0/10)]
		self.axis = Axes(x_range = x_range, y_range = y_range, x_length = 4, y_length = y_length, axis_config = {'color': BLACK, "tip_width": 0.15, "tip_height": 0.15}).move_to(ax_center)
		# self.add(self.axis)

		# ground
		ground_line = Line(start = self.axis.c2p(-2, 0, 0), end = self.axis.c2p(2, 0, 0), stroke_color = BLACK)
		ground = Polygon(self.axis.c2p(-2, 0, 0), self.axis.c2p(2, 0, 0), self.axis.c2p(2, -y0/10, 0), self.axis.c2p(-2, -y0/10, 0), stroke_opacity = 0, fill_color = BLACK, fill_opacity = 0.125)
		self.add(ground_line, ground)



	# drop
	def get_drop(self, t):
		drop_y = get_y(t)
		drop_pos = self.axis.c2p(0, drop_y, 0)

		drop_circle = Circle(radius = 0.2, stroke_color = GRAY, fill_color = GRAY, fill_opacity = 1) 
		drop_circle_inner = Circle(radius = 0.03, stroke_color = BLACK, fill_color = BLACK, fill_opacity = 1)
		return VGroup(drop_circle, drop_circle_inner).move_to(drop_pos)


	# gravitation
	def get_gravitation(self, t):
		drop_y = get_y(t)
		drop_pos = self.axis.c2p(0, drop_y, 0)

		arrow_g = get_force_gravitation(t)
		arrow_end_pos = self.axis.c2p(0, drop_y + arrow_g, 0)

		gravitation_arrow = Line(start = drop_pos, end = arrow_end_pos, color = uni_blau_2).add_tip(tip_length = 0.125, tip_width = 0.125)
		gravitation_descriptor_vec = Tex(r"$\Vec{F}_{G}=m\Vec{g}$", font_size = 32, color = uni_blau_2).next_to(gravitation_arrow.tip, 2*RIGHT)
		gravitation_descriptor = Tex(r"$F_{G}=\left|m\Vec{g}\right|$", font_size = 32, color = uni_blau_2).next_to(gravitation_arrow.tip, 2*RIGHT)
		gravitation_arrow.descriptor_vec = gravitation_descriptor_vec
		gravitation_arrow.descriptor = gravitation_descriptor
		return gravitation_arrow


	# friction
	def get_friction(self, t):
		drop_y = get_y(t)
		drop_pos = self.axis.c2p(0, drop_y, 0)

		arrow_k = get_force_friction(t)
		arrow_end_pos = self.axis.c2p(0, drop_y + arrow_k, 0)

		friction_arrow = Line(start = drop_pos, end = arrow_end_pos, color = jura_rot).add_tip(tip_length = 0.125, tip_width = 0.125)
		friction_descriptor_vec = Tex(r"$\Vec{F}_{R}=k\Vec{v}$", font_size = 32, color = jura_rot).next_to(friction_arrow.tip, 2*RIGHT) 
		friction_descriptor = Tex(r"$F_{R}=\left|k\Vec{v}\right|$", font_size = 32, color = jura_rot).next_to(friction_arrow.tip, 2*RIGHT) 
		friction_arrow.descriptor_vec = friction_descriptor_vec
		friction_arrow.descriptor = friction_descriptor
		return friction_arrow

In [ ]:
%%manim -qh --fps 60 $video_scene


class falling_drop_Scene(Scene):
	def construct(self):
		self.camera.background_color = WHITE

		# initial time
		t0 = 0

		falling_drop = FallingDrop()
		self.add(falling_drop)

		drop = falling_drop.get_drop(t0)
		self.add(drop)

		# vectors
		g_vec = falling_drop.get_gravitation(t0)
		k_vec = falling_drop.get_friction(t0)
		# self.add(g_vec, k_vec)

		# descriptors
		g_vec_descriptor_vec = g_vec.descriptor_vec
		k_vec_descriptor_vec = k_vec.descriptor_vec
		# self.add(g_vec_descriptor_vec, k_vec_descriptor_vec)

		g_vec_descriptor = g_vec.descriptor
		k_vec_descriptor = k_vec.descriptor


		# updater for everything
		def drop_updater(fdrop):
			t = time.get_value()
			new_drop = fdrop.get_drop(t)
			new_g_vec = fdrop.get_gravitation(t)
			new_k_vec = fdrop.get_friction(t)
			new_g_vec_descriptor_vec = new_g_vec.descriptor_vec
			new_k_vec_descriptor_vec = new_k_vec.descriptor_vec
			new_g_vec_descriptor = new_g_vec.descriptor
			new_k_vec_descriptor = new_k_vec.descriptor
			drop.become(new_drop)
			g_vec.become(new_g_vec)
			k_vec.become(new_k_vec)
			g_vec_descriptor_vec.become(new_g_vec_descriptor_vec)
			k_vec_descriptor_vec.become(new_k_vec_descriptor_vec)

			g_vec_descriptor.become(new_g_vec_descriptor)
			k_vec_descriptor.become(new_k_vec_descriptor)


		time = ValueTracker(0)
		t_stop = 2

		self.wait(0.5)
		self.play(FadeIn(g_vec), FadeIn(g_vec_descriptor_vec), run_time = 3)
		self.wait(1.5)
		falling_drop.add_updater(drop_updater)
		self.play(time.animate.set_value(t_stop), rate_func = linear, run_time = t_stop)
		falling_drop.remove_updater(drop_updater)
		self.wait(0.5)
		self.play(FadeIn(k_vec), FadeIn(k_vec_descriptor_vec), run_time = 3)
		self.wait(1.5)
		self.play(FadeTransform(g_vec_descriptor_vec, g_vec_descriptor), FadeTransform(k_vec_descriptor_vec, k_vec_descriptor), run_time = 3)
		self.wait(1.5)

		falling_drop.add_updater(drop_updater)
		self.play(time.animate.set_value(5), rate_func = linear, run_time = 5-t_stop)
		self.wait(1.5)   

Manim Community v0.22.1